# 04 · Semantic rule generalization

**Question:** Where does a frozen semantic encoder help or fail?

This notebook renders recorded **competition-data** evidence. It verifies the committed artifact hashes, not private out-of-fold predictions. No credentials, raw comments, weight downloads, or training are needed. Full private metric recomputation remains `uv run jigsaw review`.

The recorded encoder is **Qwen3-Embedding-0.6B**, pinned to an immutable Hub revision. Qwen weights are frozen; only the similarity classifier learns fold labels. The semantic margin uses a predeclared temperature of 0.1 and is not claimed to be calibrated.

In [1]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display
from jigsaw_rules.review import public_evidence
from jigsaw_rules.runtime import environment

root = Path(os.environ.get("JIGSAW_ROOT", Path.cwd())).resolve()
if root.name == "notebooks":
    root = root.parent
baseline = public_evidence(root, "baseline")
semantic = public_evidence(root, "semantic")
assert baseline["training_sha256"] == semantic["training_sha256"]
print("Competition data | 2,029 training rows | recorded local cross-validation")
print("Verification: aggregate file checksums and provenance; no model fitting.")

names = {"comment_only": "Comment-only TF-IDF", "rule_examples": "Rule/example TF-IDF",
         "semantic_margin": "Frozen semantic margin", "semantic_classifier": "Semantic classifier"}
protocols = {"seen_rule": "Familiar rules", "heldout_rule": "Held-out rule"}

def metric_table(records):
    return pd.DataFrame([{"Model": names[r["model"]], "Validation": protocols[r["protocol"]],
        "Rule macro AUC": r["metrics"]["rule_macro_auc"],
        "Log loss": r["metrics"]["log_loss"], "Brier": r["metrics"]["brier"],
        "Average precision": r["metrics"]["average_precision"]} for r in records]).round(4)


Competition data | 2,029 training rows | recorded local cross-validation
Verification: aggregate file checksums and provenance; no model fitting.


## Held-out behavior by rule
Aggregate improvements can conceal deterioration on individual policies. The table keeps the two observed rules separate.

In [2]:
records = baseline["results"] + semantic["results"]
per_rule = pd.DataFrame([{"Model": names[r["model"]], "Rule": rule.split(":")[0], "ROC AUC": auc}
    for r in records if r["protocol"] == "heldout_rule" for rule, auc in r["metrics"]["per_rule_auc"].items()])
display(per_rule.pivot(index="Model", columns="Rule", values="ROC AUC").round(4))

Rule,No Advertising,No legal advice
Model,,
Comment-only TF-IDF,0.6495,0.5586
Frozen semantic margin,0.7135,0.5566
Rule/example TF-IDF,0.6673,0.5639
Semantic classifier,0.6380,0.5336


## Probability diagnostics
Calibration error uses 10 equal-width bins. Precision, recall, and F1 use the fixed 0.5 diagnostic threshold; this is not a tuned deployment decision. Threshold selection and calibration fitting require nested validation.

In [3]:
diagnostics = pd.DataFrame([{"Model": names[r["model"]],
    "Pooled AUC": r["metrics"]["pooled_auc"], "Calibration error": r["metrics"]["ece_10_equal_width_bins"],
    "Precision@0.5": r["metrics"]["precision_at_0_5"], "Recall@0.5": r["metrics"]["recall_at_0_5"],
    "F1@0.5": r["metrics"]["f1_at_0_5"]} for r in records if r["protocol"] == "heldout_rule"])
display(diagnostics.round(4))

,Model,Pooled AUC,Calibration error,Precision@0.5,Recall@0.5,F1@0.5
0,Comment-only TF-IDF,0.6246,0.0366,0.5839,0.7051,0.6388
1,Rule/example TF-IDF,0.6317,0.0618,0.5693,0.8206,0.6722
2,Frozen semantic margin,0.6221,0.0607,0.5893,0.5984,0.5938
3,Semantic classifier,0.4914,0.1679,0.5199,0.3928,0.4475


## Recorded runtime and memory
These are measurements from the first successful model invocation, not the time required to render this notebook. They do not claim that earlier failed attempts cost no time. Truncation counts refer to unique encoded inputs.

In [4]:
timing = semantic["timing"]
encoder = timing["encoder"]
display(pd.DataFrame({"Measurement": ["First successful invocation (seconds)", "Encoding (seconds)", "Peak process memory (GiB)", "Unique inputs", "Truncated inputs"],
    "Value": [timing["first_completion_wall_seconds"], encoder["encode_seconds"], encoder["peak_rss_gib"], encoder["unique_texts"], encoder["truncated_rows"]]}).round(3))

,Measurement,Value
0,First successful invocation (seconds),1004.707
1,Encoding (seconds),983.690
2,Peak process memory (GiB),3.854
3,Unique inputs,1875.000
4,Truncated inputs,1.000


## What to test next
The next model experiment should jointly encode rule, comment, and support examples. Predeclare comment-only, rule-plus-comment, and positive/negative-example ablations; preserve the split registry; then compare cross-encoder or parameter-efficient fine-tuning candidates with the unchanged lexical reference. More complexity is accepted only when measured evidence supports it.

Before any paid run, approve hardware, maximum duration, and spending limits. Current embedding checkpoints resume at completed shards, and CPU training resumes at completed folds—not inside an interrupted solver. GPU optimizer-state resume is a later deliverable.

The standalone [Kaggle notebook](../kaggle/submission.ipynb) is an offline lexical reference. A preview CSV is not a scored submission, and authenticated late-scoring eligibility remains unverified.

Return to [03 · Results and decision](03_saved_results.ipynb).